# Activity 3: Charts That Argue

You have built static charts with `.plot`, interactive charts with hvPlot, and interactive charts
with Plotly Express. This notebook is not about a new library. It is about three rules that apply
to every chart you build, in any library, for the rest of this course and in the Week 8 capstone:

1. Pick the chart that answers the question. The wrong chart type can hide a finding that the right
   one shows instantly.
2. Write the title as the conclusion, not as a label for the axes.
3. Annotate the point that matters, so the reader does not have to hunt for it.

This is the short notebook, about 30 minutes. You will work with the same hospital claims data as
Activity 1 and Activity 2.

## Chart choice

Before you pick a chart type, name the shape of the question you are answering.

| Question shape | Chart | Avoid |
|---|---|---|
| How does one number compare across categories? | Bar | Pie, which makes close values indistinguishable |
| How did one number move over time? | Line | Bar, which implies discrete buckets |
| How are two numbers related? | Scatter | Dual-axis line, which invents correlations |
| How is one number distributed? | Histogram or box | A single mean, which hides the shape |
| How does a total split into parts? | Stacked bar | Pie, once you exceed about four slices |

The rest of this notebook makes the first row concrete: the same data, once as a pie chart and once
as a bar chart.

Import `pandas` for the data and `matplotlib.pyplot` for `plt.title`, `plt.annotate`, and
`plt.show`. Expect no output, the cell just runs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

Notebooks do not share state with each other, so this notebook starts from the same raw file
Activity 1 and Activity 2 used. Load the hospital claims data. Expect no visible output yet, this
just loads the DataFrame into memory.

In [ ]:
claims = pd.read_parquet("data/hospital_claims.parquet")

Rebuild the same charge to payment ratio and state aggregate from the earlier activities: charge
to payment ratio, averaged by state, highest first. Expect New Jersey at 6.65, Nevada at 6.03,
California at 5.49, Florida at 5.35, and Texas at 4.63, out of 163,065 total rows.

In [ ]:
claims["ratio"] = claims["Average Covered Charges"] / claims["Average Total Payments"]
by_state = claims.groupby("Provider State")["ratio"].mean().sort_values(ascending=False)
by_state.head()

## A chart that fails on purpose

The question here is "how does one number, the ratio, compare across categories, the states." The
chart-choice table says the answer is a bar, and to avoid a pie. Plot it as a pie anyway, so you
can see what "avoid" means instead of just being told. Expect a pie chart where the top ten slices
look almost identical: the values only run from about 6.65 down to about 3.99, and a pie chart has
no way to show a difference that small.

In [ ]:
by_state.head(10).plot(kind="pie", figsize=(7, 7), ylabel="")
plt.title("Charge to payment ratio by state")
plt.show()

Before you scroll to the next chart, look at the pie chart above. Try to rank the ten states from
the slices alone, and notice that the title, "Charge to payment ratio by state," only labels what
the chart is about. It does not tell you anything. Name for yourself why the chart fails.

The next chart plots the exact same `by_state` values as a sorted horizontal bar. Expect the
ranking to be instantly readable: New Jersey clearly the longest bar, decreasing down to South
Carolina, with bar lengths you can compare at a glance where the pie slices looked the same.

In [ ]:
by_state.head(10).plot(
    kind="barh",
    title="New Jersey hospitals bill 6.7 times what Medicare pays, Texas 4.6",
    xlabel="Average charge to payment ratio",
    ylabel="",
    figsize=(10, 5),
    color="#B31B1B",
    legend=False,
)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

A conclusion title must be checkable against the chart it sits on. This title states both New
Jersey's ratio and Texas's ratio directly, so you can look at the two bars and confirm both numbers
without doing any math.

Avoid phrasing like "New Jersey bills N times more than Texas." Readers split on whether that means
N times as much or N times greater than, and a title that different readers parse differently is
not a fact you can check, it is a puzzle. State both values, the way the title above does, or say
the comparison as a percentage if you want a single number.

The verified numbers: New Jersey 6.65, Texas 4.63, so New Jersey is about 1.44 times Texas, which
is about 44 percent higher.

## Titles as conclusions

A title has room for one sentence. Spend it on the finding, not on what the axes already show.

| Label title | Conclusion title |
|---|---|
| Ratio by state | New Jersey hospitals bill 6.7 times what Medicare pays |
| Discharges over time | Discharge volume is flat, so the cost growth is price, not demand |
| Charges versus payments | Payments barely move as charges rise |

Every chart in this notebook follows the right-hand column: the title states the finding first, and
the axis labels carry the units.

## Annotate the point that matters

Maryland's ratio, 1.06, is not just the lowest in the country, it is far enough below the rest that
it deserves a label directly on the chart rather than leaving the reader to spot it in a legend or
an axis. Plot the ten lowest-ratio states, sorted ascending, then use `ax.annotate` to draw an
arrow from an explanation to Maryland's bar. Expect the same style of horizontal bar chart, with
Maryland at the bottom of the sorted order, and an arrow pointing at Maryland's bar with a one-line
explanation next to it.

In [ ]:
ax = by_state.tail(10).sort_values().plot(
    kind="barh",
    title="Maryland is the exception, and the reason is policy",
    xlabel="Average charge to payment ratio",
    ylabel="",
    figsize=(10, 5),
    color="#2C6E49",
    legend=False,
)
ax.annotate(
    "Maryland sets hospital rates for all payers",
    xy=(1.06, 0),
    xytext=(2.2, 1.5),
    arrowprops={"arrowstyle": "->"},
)
plt.tight_layout()
plt.show()

## The six-step story arc

Every deliverable in this course, from a case study to a dashboard, follows the same six steps.
Learn the order now. You will reuse it later this week and in the Week 8 capstone.

1. **Decision.** What choice is this analysis meant to inform?
2. **Context.** What does the audience already know, and what do they need to know to follow the
   rest?
3. **Evidence.** What is the specific finding, in numbers?
4. **Insight.** Why does that finding matter, what does it mean for the decision?
5. **Caveat.** What does the data not show, stated honestly?
6. **Recommendation.** What should the audience do next?

**Worked example, applied to the Maryland finding:**

- **Decision.** Should a payer negotiate hospital rates directly, the way Maryland does, or let the
  market set them the way the rest of the country does?
- **Context.** In every other state in this dataset, each hospital negotiates its own charges
  against what Medicare pays.
- **Evidence.** Maryland's charge to payment ratio is 1.06, the lowest of any state. The next
  lowest, Vermont, is 1.75, and the range runs up to New Jersey at 6.65.
- **Insight.** Maryland is not an outlier of scale, it is an outlier of policy. The state sets
  hospital rates for all payers, which keeps the gap between what is billed and what is paid nearly
  closed.
- **Caveat.** This is Medicare inpatient claims only. It says nothing about outpatient care,
  commercial insurance, or whether Maryland's model affects patient access or wait times.
- **Recommendation.** Before a payer or state health department expands Maryland-style rate setting
  elsewhere, study its effect on hospital margins and patient access in Maryland itself. A lower
  ratio alone does not prove the policy is good for patients.

## Your turn

Pick any single chart you built this morning, in this notebook, Activity 1, or Activity 2. Write
three sentences:

1. A conclusion title for that chart: the finding it proves.
2. One sentence of caveat: name something specific the chart does not show.
3. One sentence of recommendation: what someone should do with this finding.

The deliverable is the three sentences, not a new chart. Write them in the markdown cell below. If
you want to redraw the chart first, use the empty code cell after it.

_Your answer here._